# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedwaqasahmad/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I'm choosing a Random Forest classifier. Notebook 01 already showed a random forest beating a hand-written rule by roughly 3x on Precision@50 on this exact dataset, so there's precedent it works well here. It also handles the mix of numeric and categorical features I built in ML-05 without needing heavy preprocessing, and it's less prone to overfitting than a single decision tree while staying reasonably fast to train.

In [5]:
import os, subprocess
REPO_URL = "https://github.com/syedwaqasahmad/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_churned"] = df["trend_direction"].str.lower().eq("down").astype(int)

numeric_features = ["impressions_90d","search_volume","competition","cpc","word_count","char_count",
                     "days_with_impressions","days_with_sessions","content_age_days",
                     "days_since_last_update","ctr","avg_position","engagement_rate",
                     "scroll_rate","ai_traffic_pct"]
categorical_features = ["competition_level","content_type","main_intent","age_tier",
                         "freshness_tier","word_count_tier","impression_tier","position_tier"]

X_numeric = df[numeric_features].replace([np.inf, -np.inf], np.nan).fillna(0)
X_categorical = pd.get_dummies(df[categorical_features], dummy_na=True)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["is_churned"]
print("Method: Random Forest. Feature matrix:", X.shape)

Method: Random Forest. Feature matrix: (30000, 54)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Using a client-holdout split — grouped by client_id, not a random row split. This matters because a random split could put the same client's pages in both train and test, letting the model "cheat" by memorizing client-specific patterns rather than learning generalizable signal. This matches the approach the real starter pipeline (scripts/03_train_model.py) uses.

In [6]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(df["client_id"].iloc[train_idx])
test_clients = set(df["client_id"].iloc[test_idx])
print(f"Train: {len(X_train)} rows, {len(train_clients)} clients")
print(f"Test: {len(X_test)} rows, {len(test_clients)} clients")
print("Overlap between train/test clients:", len(train_clients & test_clients))

Train: 22885 rows, 24 clients
Test: 7115 rows, 8 clients
Overlap between train/test clients: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Training the Random Forest on the train split, then comparing its Precision@20 and Precision@50 on the test split against my Week-4 baseline score, using the exact same test rows and the exact same metric.

In [7]:
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced")
model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:, 1]

baseline_test = df.iloc[test_idx].copy()
reason_stale_visible = ((baseline_test["days_since_last_update"] >= 180) & (baseline_test["impressions_90d"] >= 500)).astype(int)
reason_declining_demand = ((baseline_test["trend_direction"].str.lower() == "down") & (baseline_test["impressions_90d"] >= 100)).astype(int)
reason_position_decay = ((baseline_test["avg_position"] <= 10) & (baseline_test["content_age_days"] >= 180)).astype(int)
baseline_scores = 0.5*reason_stale_visible + 0.3*reason_declining_demand + 0.2*reason_position_decay

results = []
for k in (20, 50):
    results.append({
        "k": k,
        "baseline_precision": precision_at_k(baseline_scores.values, y_test.values, k),
        "model_precision": precision_at_k(model_scores, y_test.values, k)
    })
print(pd.DataFrame(results))

    k  baseline_precision  model_precision
0  20                 1.0             0.70
1  50                 1.0             0.68


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Looking at which test-set pages the model got most wrong (highest confidence, wrong prediction), and which features it leans on most (feature importances) to understand what it actually learned versus what my hand-written rule assumed.

In [8]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 10 most important features:")
print(importances.head(10))

test_df = df.iloc[test_idx].copy()
test_df["model_score"] = model_scores
test_df["predicted"] = (model_scores >= 0.5).astype(int)
wrong = test_df[test_df["predicted"] != test_df["is_churned"]].sort_values("model_score", ascending=False)
print(f"\nMisclassified test rows: {len(wrong)} of {len(test_df)}")
print(wrong[["content_id","model_score","is_churned","impressions_90d","trend_direction"]].head(5))

Top 10 most important features:
impressions_90d          0.110059
avg_position             0.103442
days_with_impressions    0.089562
content_age_days         0.075560
days_with_sessions       0.066504
ctr                      0.062808
word_count               0.060916
char_count               0.059042
scroll_rate              0.058991
search_volume            0.033194
dtype: float64

Misclassified test rows: 3034 of 7115
                 content_id  model_score  is_churned  impressions_90d  \
22042  content_2ba626fea4d6        0.975           0              360   
10080  content_35d63627bf3e        0.960           0             1525   
22526  content_1d0963b56227        0.960           0             3445   
22461  content_7e3be2e230f5        0.950           0              909   
27993  content_26d48a980581        0.940           0             1266   

      trend_direction  
22042              up  
10080          stable  
22526              up  
22461          stable  
27993          

Looking at which test-set pages the model got most wrong, and which features it leans on most, to understand what it learned versus what my hand-written rule assumed.

Important catch: my baseline's "declining_with_demand" reason code directly uses trend_direction == "down" — the same field my label is built from. So the baseline's perfect Precision@20/50 (1.0) is partly circular, not a genuine predictive win — it's restating the answer for rows where that code fires. The model's 0.70/0.68 is the honest, comparable number, since it never sees trend_direction directly. A fairer baseline comparison would drop declining_with_demand entirely and use only stale_visible_page + page_one_decay_risk.

In [9]:
# Fair baseline: drop the circular reason code
fair_baseline_scores = 0.5*reason_stale_visible + 0.2*reason_position_decay
for k in (20, 50):
    print(f"Fair baseline Precision@{k}: {precision_at_k(fair_baseline_scores.values, y_test.values, k):.3f}")
    print(f"Model Precision@{k}: {precision_at_k(model_scores, y_test.values, k):.3f}")

Fair baseline Precision@20: 0.750
Model Precision@20: 0.700
Fair baseline Precision@50: 0.660
Model Precision@50: 0.680


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.